**### Очередная попытка реализации RAG, потому что мне не хватает ума ее сделать адекватно с первого раза.**

я на гпу Л4 делала просто потому что мне он нравится и он еще ни разу меня не подводил


## Этап 1
Идея такая: документы-очистка+чанкинг-эмбеддинги-фаисс/хрома пока думаю-ретриверы-оценка ретривера-локальная ллм-раг-оценка качества-оптимизация (с божьей помочью).

Сначала устанавливаем необходимые библиотеки для подготовки окружения.

In [1]:
!pip install -q langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


Колаб на реквесты ругался, так что их устанавливаю отдельно и перезапускаю все это дело.

In [2]:
!pip install -q requests==2.32.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 8.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.1 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


Я честно скажу просто надеюсь, что запустится и так, у меня нет сил и желания с этим бороться. Сейчас проверим.

In [3]:
import requests
print(requests.__version__)

2.32.4


Отлично, пока все работает. Теперь подтягиваем импорты.

In [4]:
import re
import pandas as pd
from langchain_core.documents import Document

#разные стратегии чанкинга
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

#импортируем HF-эмбеддинги и FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

Теперь нужно создать синтетические документы. Пять штук о банковских продуктах с единым форматом. Я, честно говоря, сильно не заморачивалась, так что реализовала этот момент очень просто.

Формат единый: айди, название, тип продукта, сурс и текст. Все.

In [5]:
raw_documents = [
    {
        "doc_id": "credit_cash_001",
        "title": "Потребительский кредит",
        "product_type": "credit",
        "source": "synthetic_bank_docs",
        "text": """
        Потребительский кредит «Универсальный» предоставляется физическим лицам на любые цели.
        Минимальная сумма кредита составляет 50 000 рублей, максимальная сумма — 3 000 000 рублей.
        Срок кредитования — от 12 до 60 месяцев.
        Процентная ставка начинается от 12,9% годовых и зависит от кредитной истории клиента,
        уровня дохода и наличия страхования. При отказе от страхования ставка может быть увеличена на 3 процентных пункта.
        Досрочное погашение возможно без комиссии. Заявка рассматривается от 5 минут до 2 рабочих дней.
        """
    },
    {
        "doc_id": "mortgage_001",
        "title": "Ипотечный кредит",
        "product_type": "mortgage",
        "source": "synthetic_bank_docs",
        "text": """
        Ипотечный кредит предоставляется на покупку квартиры, апартаментов или жилого дома.
        Минимальная сумма ипотеки составляет 500 000 рублей, максимальная сумма — 30 000 000 рублей.
        Первоначальный взнос — от 15% стоимости недвижимости.
        Срок кредитования — от 3 до 30 лет.
        Базовая процентная ставка составляет от 10,5% годовых.
        Для зарплатных клиентов действует скидка 0,5 процентного пункта.
        Обязательным условием является страхование объекта недвижимости.
        Страхование жизни и здоровья заёмщика не является обязательным, но влияет на итоговую ставку.
        """
    },
    {
        "doc_id": "deposit_001",
        "title": "Депозитные продукты",
        "product_type": "deposit",
        "source": "synthetic_bank_docs",
        "text": """
        Банк предлагает вклад «Надёжный доход» для физических лиц.
        Минимальная сумма вклада составляет 10 000 рублей.
        Срок размещения средств — 3, 6, 12 или 24 месяца.
        Процентная ставка зависит от срока вклада: 7,5% годовых на 3 месяца,
        8,2% годовых на 6 месяцев, 9,0% годовых на 12 месяцев и 8,7% годовых на 24 месяца.
        Пополнение возможно в течение первых 30 дней после открытия вклада.
        Частичное снятие не предусмотрено.
        Проценты могут выплачиваться ежемесячно или капитализироваться.
        """
    },
    {
        "doc_id": "borrower_requirements_001",
        "title": "Требования к заёмщикам",
        "product_type": "borrower_requirements",
        "source": "synthetic_bank_docs",
        "text": """
        Заёмщиком может быть гражданин Российской Федерации в возрасте от 21 года до 70 лет на дату окончания кредита.
        Для оформления кредита необходим паспорт, СНИЛС и документ, подтверждающий доход.
        Минимальный трудовой стаж на текущем месте работы должен составлять не менее 3 месяцев.
        Общий трудовой стаж должен быть не менее 1 года.
        Для суммы кредита свыше 1 500 000 рублей банк может запросить копию трудовой книжки
        или выписку из электронной трудовой книжки.
        Клиент должен иметь постоянную или временную регистрацию на территории региона присутствия банка.
        """
    },
    {
        "doc_id": "faq_001",
        "title": "FAQ по банковским продуктам",
        "product_type": "faq",
        "source": "synthetic_bank_docs",
        "text": """
        Часто задаваемые вопросы.
        Можно ли досрочно погасить кредит? Да, досрочное погашение доступно без комиссии.
        Можно ли открыть вклад онлайн? Да, вклад можно открыть в мобильном приложении или интернет-банке.
        Что будет при просрочке платежа по кредиту? При просрочке начисляется неустойка 0,1% от суммы просроченного платежа за каждый день.
        Можно ли изменить дату платежа? Да, клиент может один раз в год изменить дату ежемесячного платежа.
        Как получить справку об остатке задолженности? Справку можно скачать в интернет-банке или получить в офисе банка.
        """
    }
]

len(raw_documents)

5

Следующим этапом чистим текст, тут в подробности вдаваться смысла не вижу и так понятно, что будем делать.

In [6]:
def clean_text(text):
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s.,;:!?%№()/\-–—«»]", "", text)
    return text.strip()

cleaned_documents = []

for doc in raw_documents:
    cleaned_doc = doc.copy()
    cleaned_doc["text"] = clean_text(doc["text"])
    cleaned_documents.append(cleaned_doc)

print(cleaned_documents[0]["text"])

Потребительский кредит «Универсальный» предоставляется физическим лицам на любые цели. Минимальная сумма кредита составляет 50 000 рублей, максимальная сумма — 3 000 000 рублей. Срок кредитования — от 12 до 60 месяцев. Процентная ставка начинается от 12,9% годовых и зависит от кредитной истории клиента, уровня дохода и наличия страхования. При отказе от страхования ставка может быть увеличена на 3 процентных пункта. Досрочное погашение возможно без комиссии. Заявка рассматривается от 5 минут до 2 рабочих дней.


Теперь было бы неплохо это все перевести в формат LangChain документа, чтобы с ним дальше работать.

Метаданные нам будут нужны для фильтрации и цитирования.

In [7]:
lc_documents = []

for doc in cleaned_documents:
    lc_documents.append(
        Document(
            page_content=doc["text"],
            metadata={
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "product_type": doc["product_type"],
                "source": doc["source"]
            }
        )
    )


print(f"Количество документов: {len(lc_documents)}")

lc_documents[0]

Количество документов: 5


Document(metadata={'doc_id': 'credit_cash_001', 'title': 'Потребительский кредит', 'product_type': 'credit', 'source': 'synthetic_bank_docs'}, page_content='Потребительский кредит «Универсальный» предоставляется физическим лицам на любые цели. Минимальная сумма кредита составляет 50 000 рублей, максимальная сумма — 3 000 000 рублей. Срок кредитования — от 12 до 60 месяцев. Процентная ставка начинается от 12,9% годовых и зависит от кредитной истории клиента, уровня дохода и наличия страхования. При отказе от страхования ставка может быть увеличена на 3 процентных пункта. Досрочное погашение возможно без комиссии. Заявка рассматривается от 5 минут до 2 рабочих дней.')

Далее, как указано по заданию, реализовываем три варианта чанкинга. С фиксированным размером, по предложениям и рекурсивный, ничег овыдумывать не буду.

PS я обожаю делать кастомные чанкеры, у меня есть классный для моей ВКР.

In [8]:
#фиксированный размер чанков
size_splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separator=" "
)

chunks_size = size_splitter.split_documents(lc_documents)

for i, chunk in enumerate(chunks_size):
    chunk.metadata["chunk_strategy"] = "size"
    chunk.metadata["chunk_id"] = f"size_{i}"

print(f"Количество чанков по размеру: {len(chunks_size)}")
print(chunks_size[0])

Количество чанков по размеру: 9
page_content='Потребительский кредит «Универсальный» предоставляется физическим лицам на любые цели. Минимальная сумма кредита составляет 50 000 рублей, максимальная сумма — 3 000 000 рублей. Срок кредитования — от 12 до 60 месяцев. Процентная ставка начинается от 12,9% годовых и зависит от кредитной истории клиента, уровня дохода и наличия страхования. При отказе от страхования ставка может быть увеличена на 3 процентных пункта. Досрочное погашение возможно без комиссии. Заявка рассматривается от 5 минут до' metadata={'doc_id': 'credit_cash_001', 'title': 'Потребительский кредит', 'product_type': 'credit', 'source': 'synthetic_bank_docs', 'chunk_strategy': 'size', 'chunk_id': 'size_0'}


In [9]:
#чанкер по предложениям
def sentence_chunk_documents(documents, max_sentences=3):
    chunks = []

    for doc in documents:
        # Делим текст на предложения по точкам, вопросительным и восклицательным знакам.
        sentences = re.split(r"(?<=[.!?])\s+", doc.page_content)

        # Объединяем предложения группами.
        for i in range(0, len(sentences), max_sentences):
            chunk_text = " ".join(sentences[i:i + max_sentences]).strip()

            if chunk_text:
                chunks.append(
                    Document(
                        page_content=chunk_text,
                        metadata={
                            **doc.metadata,
                            "chunk_strategy": "sentence",
                            "chunk_id": f"{doc.metadata['doc_id']}_sent_{i // max_sentences}"
                        }
                    )
                )

    return chunks


chunks_sentence = sentence_chunk_documents(lc_documents, max_sentences=3)

print(f"Количество чанков по предложениям: {len(chunks_sentence)}")
print(chunks_sentence[0])

Количество чанков по предложениям: 15
page_content='Потребительский кредит «Универсальный» предоставляется физическим лицам на любые цели. Минимальная сумма кредита составляет 50 000 рублей, максимальная сумма — 3 000 000 рублей. Срок кредитования — от 12 до 60 месяцев.' metadata={'doc_id': 'credit_cash_001', 'title': 'Потребительский кредит', 'product_type': 'credit', 'source': 'synthetic_bank_docs', 'chunk_strategy': 'sentence', 'chunk_id': 'credit_cash_001_sent_0'}


In [10]:
#рекурсивный чанкинг
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks_recursive = recursive_splitter.split_documents(lc_documents)

for i, chunk in enumerate(chunks_recursive):
    chunk.metadata["chunk_strategy"] = "recursive"
    chunk.metadata["chunk_id"] = f"recursive_{i}"

print(f"Количество рекурсивных чанков: {len(chunks_recursive)}")
print(chunks_recursive[0])

Количество рекурсивных чанков: 9
page_content='Потребительский кредит «Универсальный» предоставляется физическим лицам на любые цели. Минимальная сумма кредита составляет 50 000 рублей, максимальная сумма — 3 000 000 рублей. Срок кредитования — от 12 до 60 месяцев. Процентная ставка начинается от 12,9% годовых и зависит от кредитной истории клиента, уровня дохода и наличия страхования. При отказе от страхования ставка может быть увеличена на 3 процентных пункта. Досрочное погашение возможно без комиссии' metadata={'doc_id': 'credit_cash_001', 'title': 'Потребительский кредит', 'product_type': 'credit', 'source': 'synthetic_bank_docs', 'chunk_strategy': 'recursive', 'chunk_id': 'recursive_0'}


Чтобы сравнить три стратегии чанкеров (хотя понятно, что рекурсивный лучший, но не суть) собираем статистику: количество, средняя длина, мин и макс длины.

Это все первичный анализ, просто смотрим пока.

In [11]:
def describe_chunks(chunks, strategy_name):
    lengths = [len(chunk.page_content) for chunk in chunks]

    return {
        "strategy": strategy_name,
        "num_chunks": len(chunks),
        "avg_length": round(sum(lengths) / len(lengths), 2),
        "min_length": min(lengths),
        "max_length": max(lengths)
    }


chunk_stats = pd.DataFrame([
    describe_chunks(chunks_size, "size"),
    describe_chunks(chunks_sentence, "sentence"),
    describe_chunks(chunks_recursive, "recursive")
])

chunk_stats

,strategy,num_chunks,avg_length,min_length,max_length
0,size,9,337.33,116,499
1,sentence,15,175.60,52,280
2,recursive,9,316.89,97,478


Ну и посмотрим примеры.

In [12]:
def print_chunk_examples(chunks, strategy_name, n=3):
    print(f"\n=== Примеры чанков: {strategy_name} ===\n")

    for i, chunk in enumerate(chunks[:n], start=1):
        print(f"Чанк {i}")
        print("Метаданные:", chunk.metadata)
        print("Текст:", chunk.page_content)
        print("-" * 100)


print_chunk_examples(chunks_size, "size")
print_chunk_examples(chunks_sentence, "sentence")
print_chunk_examples(chunks_recursive, "recursive")


=== Примеры чанков: size ===

Чанк 1
Метаданные: {'doc_id': 'credit_cash_001', 'title': 'Потребительский кредит', 'product_type': 'credit', 'source': 'synthetic_bank_docs', 'chunk_strategy': 'size', 'chunk_id': 'size_0'}
Текст: Потребительский кредит «Универсальный» предоставляется физическим лицам на любые цели. Минимальная сумма кредита составляет 50 000 рублей, максимальная сумма — 3 000 000 рублей. Срок кредитования — от 12 до 60 месяцев. Процентная ставка начинается от 12,9% годовых и зависит от кредитной истории клиента, уровня дохода и наличия страхования. При отказе от страхования ставка может быть увеличена на 3 процентных пункта. Досрочное погашение возможно без комиссии. Заявка рассматривается от 5 минут до
----------------------------------------------------------------------------------------------------
Чанк 2
Метаданные: {'doc_id': 'credit_cash_001', 'title': 'Потребительский кредит', 'product_type': 'credit', 'source': 'synthetic_bank_docs', 'chunk_strategy': 'size', '

Чтобы подобрать чанкер, хорошо бы еще подобрать модель эмбеддингов. В данном случае я использую multilingual-e5-base с нормализованными векторами, что будет ползено для similarity  search.

In [13]:
embedding_model_name = "intfloat/multilingual-e5-base"

embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

print(f"{embedding_model_name}")

/tmp/ipykernel_10517/3497860453.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

intfloat/multilingual-e5-base


Далее будем создавать отдельную FAISS базу для каждой стретегии чанкинга, чо поможет нам потом сравнить, какая из них лучше извлекает текст.

In [14]:
vectorstore_size = FAISS.from_documents(
    documents=chunks_size,
    embedding=embedding_model
)

vectorstore_sentence = FAISS.from_documents(
    documents=chunks_sentence,
    embedding=embedding_model
)

vectorstore_recursive = FAISS.from_documents(
    documents=chunks_recursive,
    embedding=embedding_model
)

print("FAISS-базы успешно созданы.")

FAISS-базы успешно созданы.


Тепероь сохраним все локально, чтобы потом можно было скачать или на диск сохранить, в общем НА ВСЯКИЙ.

In [15]:
vectorstore_size.save_local("faiss_bank_size")
vectorstore_sentence.save_local("faiss_bank_sentence")
vectorstore_recursive.save_local("faiss_bank_recursive")

print("FAISS-базы сохранены:")
print("- faiss_bank_size")
print("- faiss_bank_sentence")
print("- faiss_bank_recursive")

FAISS-базы сохранены:
- faiss_bank_size
- faiss_bank_sentence
- faiss_bank_recursive


Быстренько проверяем similarity check, просто смотрим возвращает ли база релевантные чанки, тут как бы кто его знает.

In [16]:
test_query = "Какая максимальная сумма потребительского кредита?"

results = vectorstore_recursive.similarity_search(test_query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"Результат {i}")
    print("Метаданные:", doc.metadata)
    print("Текст:", doc.page_content)
    print("-" * 100)

Результат 1
Метаданные: {'doc_id': 'credit_cash_001', 'title': 'Потребительский кредит', 'product_type': 'credit', 'source': 'synthetic_bank_docs', 'chunk_strategy': 'recursive', 'chunk_id': 'recursive_0'}
Текст: Потребительский кредит «Универсальный» предоставляется физическим лицам на любые цели. Минимальная сумма кредита составляет 50 000 рублей, максимальная сумма — 3 000 000 рублей. Срок кредитования — от 12 до 60 месяцев. Процентная ставка начинается от 12,9% годовых и зависит от кредитной истории клиента, уровня дохода и наличия страхования. При отказе от страхования ставка может быть увеличена на 3 процентных пункта. Досрочное погашение возможно без комиссии
----------------------------------------------------------------------------------------------------
Результат 2
Метаданные: {'doc_id': 'mortgage_001', 'title': 'Ипотечный кредит', 'product_type': 'mortgage', 'source': 'synthetic_bank_docs', 'chunk_strategy': 'recursive', 'chunk_id': 'recursive_2'}
Текст: Ипотечный кредит п

И вот теперь можно уже сделать предварительное сравнение чанкеров.

Метрики будут Hit Rate@k, MRR, но они считаются дальше по коду.

In [17]:
def compare_chunking_retrieval(query, k=2):
    stores = {
        "size": vectorstore_size,
        "sentence": vectorstore_sentence,
        "recursive": vectorstore_recursive
    }

    for strategy_name, store in stores.items():
        print(f"\n=== Strategy: {strategy_name} ===")

        results = store.similarity_search(query, k=k)

        for i, doc in enumerate(results, start=1):
            print(f"\nРезультат {i}")
            print("doc_id:", doc.metadata.get("doc_id"))
            print("title:", doc.metadata.get("title"))
            print("chunk_id:", doc.metadata.get("chunk_id"))
            print("text:", doc.page_content)
            print("-" * 100)


compare_chunking_retrieval(
    query="Можно ли досрочно погасить кредит без комиссии?",
    k=2
)


=== Strategy: size ===

Результат 1
doc_id: faq_001
title: FAQ по банковским продуктам
chunk_id: size_7
text: Часто задаваемые вопросы. Можно ли досрочно погасить кредит? Да, досрочное погашение доступно без комиссии. Можно ли открыть вклад онлайн? Да, вклад можно открыть в мобильном приложении или интернет-банке. Что будет при просрочке платежа по кредиту? При просрочке начисляется неустойка 0,1% от суммы просроченного платежа за каждый день. Можно ли изменить дату платежа? Да, клиент может один раз в год изменить дату ежемесячного платежа. Как получить справку об остатке задолженности? Справку можно
----------------------------------------------------------------------------------------------------

Результат 2
doc_id: credit_cash_001
title: Потребительский кредит
chunk_id: size_1
text: 3 процентных пункта. Досрочное погашение возможно без комиссии. Заявка рассматривается от 5 минут до 2 рабочих дней.
----------------------------------------------------------------------------------

Ну более менее это все выглядит.

##Этап 2

Делаем т.н. систему ретрива. Начнем с базового similarity search на базе рекурсивных чанков.

In [18]:
query = "Какая максимальная сумма потребительского кредита?"

similarity_results = vectorstore_recursive.similarity_search(query, k=3)

for i, doc in enumerate(similarity_results, start=1):
    print(f"Результат {i}")
    print("doc_id:", doc.metadata.get("doc_id"))
    print("title:", doc.metadata.get("title"))
    print("product_type:", doc.metadata.get("product_type"))
    print("chunk_id:", doc.metadata.get("chunk_id"))
    print("text:", doc.page_content)
    print("-" * 100)

Результат 1
doc_id: credit_cash_001
title: Потребительский кредит
product_type: credit
chunk_id: recursive_0
text: Потребительский кредит «Универсальный» предоставляется физическим лицам на любые цели. Минимальная сумма кредита составляет 50 000 рублей, максимальная сумма — 3 000 000 рублей. Срок кредитования — от 12 до 60 месяцев. Процентная ставка начинается от 12,9% годовых и зависит от кредитной истории клиента, уровня дохода и наличия страхования. При отказе от страхования ставка может быть увеличена на 3 процентных пункта. Досрочное погашение возможно без комиссии
----------------------------------------------------------------------------------------------------
Результат 2
doc_id: mortgage_001
title: Ипотечный кредит
product_type: mortgage
chunk_id: recursive_2
text: Ипотечный кредит предоставляется на покупку квартиры, апартаментов или жилого дома. Минимальная сумма ипотеки составляет 500 000 рублей, максимальная сумма — 30 000 000 рублей. Первоначальный взнос — от 15% стоимос

Посмотрим еще не просто найденные документы, но и score. В FAISS score является расстоянием, а потому чем меньше score, тем ближе документ к запросу.

In [19]:
query = "Какая максимальная сумма потребительского кредита?"

results_with_scores = vectorstore_recursive.similarity_search_with_score(query, k=3)

for i, (doc, score) in enumerate(results_with_scores, start=1):
    print(f"Результат {i}")
    print("score:", score)
    print("doc_id:", doc.metadata.get("doc_id"))
    print("title:", doc.metadata.get("title"))
    print("text:", doc.page_content)
    print("-" * 100)

Результат 1
score: 0.25323272
doc_id: credit_cash_001
title: Потребительский кредит
text: Потребительский кредит «Универсальный» предоставляется физическим лицам на любые цели. Минимальная сумма кредита составляет 50 000 рублей, максимальная сумма — 3 000 000 рублей. Срок кредитования — от 12 до 60 месяцев. Процентная ставка начинается от 12,9% годовых и зависит от кредитной истории клиента, уровня дохода и наличия страхования. При отказе от страхования ставка может быть увеличена на 3 процентных пункта. Досрочное погашение возможно без комиссии
----------------------------------------------------------------------------------------------------
Результат 2
score: 0.33117142
doc_id: mortgage_001
title: Ипотечный кредит
text: Ипотечный кредит предоставляется на покупку квартиры, апартаментов или жилого дома. Минимальная сумма ипотеки составляет 500 000 рублей, максимальная сумма — 30 000 000 рублей. Первоначальный взнос — от 15% стоимости недвижимости. Срок кредитования — от 3 до 30 лет.

Видим, что ретривер ище не по точному совпадению слов, а по смысловой близости текста.

Далее посмотрим, что нам покажет MRRретривер. Тут идея вернуть не только пожие, но и разнообразные документы, что полезно, если чанки похожи друг на друга.

In [20]:
mmr_retriever = vectorstore_recursive.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10
    }
)

query = "Какие условия по ипотеке?"

mmr_results = mmr_retriever.invoke(query)

for i, doc in enumerate(mmr_results, start=1):
    print(f"MMR результат {i}")
    print("doc_id:", doc.metadata.get("doc_id"))
    print("title:", doc.metadata.get("title"))
    print("product_type:", doc.metadata.get("product_type"))
    print("text:", doc.page_content)
    print("-" * 100)

MMR результат 1
doc_id: mortgage_001
title: Ипотечный кредит
product_type: mortgage
text: Ипотечный кредит предоставляется на покупку квартиры, апартаментов или жилого дома. Минимальная сумма ипотеки составляет 500 000 рублей, максимальная сумма — 30 000 000 рублей. Первоначальный взнос — от 15% стоимости недвижимости. Срок кредитования — от 3 до 30 лет. Базовая процентная ставка составляет от 10,5% годовых. Для зарплатных клиентов действует скидка 0,5 процентного пункта. Обязательным условием является страхование объекта недвижимости
----------------------------------------------------------------------------------------------------
MMR результат 2
doc_id: borrower_requirements_001
title: Требования к заёмщикам
product_type: borrower_requirements
text: . Клиент должен иметь постоянную или временную регистрацию на территории региона присутствия банка.
----------------------------------------------------------------------------------------------------
MMR результат 3
doc_id: mortgage_00

Проведем качествуенное сравнение без метрик по ретриверам

In [21]:
def compare_similarity_and_mmr(query, k=3):
    print("QUERY:", query)
    print("\n" + "=" * 120)
    print("SIMILARITY SEARCH")
    print("=" * 120)

    similarity_docs = vectorstore_recursive.similarity_search(query, k=k)

    for i, doc in enumerate(similarity_docs, start=1):
        print(f"\nSimilarity result {i}")
        print("doc_id:", doc.metadata.get("doc_id"))
        print("title:", doc.metadata.get("title"))
        print("chunk_id:", doc.metadata.get("chunk_id"))
        print(doc.page_content)

    print("\n" + "=" * 120)
    print("MMR SEARCH")
    print("=" * 120)

    mmr_docs = mmr_retriever.invoke(query)

    for i, doc in enumerate(mmr_docs, start=1):
        print(f"\nMMR result {i}")
        print("doc_id:", doc.metadata.get("doc_id"))
        print("title:", doc.metadata.get("title"))
        print("chunk_id:", doc.metadata.get("chunk_id"))
        print(doc.page_content)

compare_similarity_and_mmr(
    query="Можно ли досрочно погасить кредит и какие условия по ставке?",
    k=3
)

QUERY: Можно ли досрочно погасить кредит и какие условия по ставке?

SIMILARITY SEARCH

Similarity result 1
doc_id: faq_001
title: FAQ по банковским продуктам
chunk_id: recursive_7
Часто задаваемые вопросы. Можно ли досрочно погасить кредит? Да, досрочное погашение доступно без комиссии. Можно ли открыть вклад онлайн? Да, вклад можно открыть в мобильном приложении или интернет-банке. Что будет при просрочке платежа по кредиту? При просрочке начисляется неустойка 0,1% от суммы просроченного платежа за каждый день. Можно ли изменить дату платежа? Да, клиент может один раз в год изменить дату ежемесячного платежа

Similarity result 2
doc_id: credit_cash_001
title: Потребительский кредит
chunk_id: recursive_1
. Досрочное погашение возможно без комиссии. Заявка рассматривается от 5 минут до 2 рабочих дней.

Similarity result 3
doc_id: borrower_requirements_001
title: Требования к заёмщикам
chunk_id: recursive_5
Заёмщиком может быть гражданин Российской Федерации в возрасте от 21 года до 70 

Ну как бы ок, пойдет, первые результаты ближе всего, но посмотрим еще BM25.

Это ключевой поик, то есть он уже не по близости векторов изет, а по совпадению слов, что полезно, если у нас по тексту находятся точные термины формата СНИЛС и пр. (я забыла подцпить библиотеку)

In [22]:
!pip install -q rank_bm25
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(chunks_recursive)
bm25_retriever.k = 3

query = "Какая неустойка при просрочке платежа?"

bm25_results = bm25_retriever.invoke(query)

for i, doc in enumerate(bm25_results, start=1):
    print(f"BM25 результат {i}")
    print("doc_id:", doc.metadata.get("doc_id"))
    print("title:", doc.metadata.get("title"))
    print("text:", doc.page_content)
    print("-" * 100)

BM25 результат 1
doc_id: faq_001
title: FAQ по банковским продуктам
text: Часто задаваемые вопросы. Можно ли досрочно погасить кредит? Да, досрочное погашение доступно без комиссии. Можно ли открыть вклад онлайн? Да, вклад можно открыть в мобильном приложении или интернет-банке. Что будет при просрочке платежа по кредиту? При просрочке начисляется неустойка 0,1% от суммы просроченного платежа за каждый день. Можно ли изменить дату платежа? Да, клиент может один раз в год изменить дату ежемесячного платежа
----------------------------------------------------------------------------------------------------
BM25 результат 2
doc_id: faq_001
title: FAQ по банковским продуктам
text: . Можно ли изменить дату платежа? Да, клиент может один раз в год изменить дату ежемесячного платежа. Как получить справку об остатке задолженности? Справку можно скачать в интернет-банке или получить в офисе банка.
--------------------------------------------------------------------------------------------------

Ну как бы четно говоря результат такое, поэтом у хочется попробовать какой-нибудь гибридный метод сделать? Получить the best of both worlds.

Идея: берём результаты BM25 и vector search, затем объединяем их без дублей.

In [23]:
vector_retriever = vectorstore_recursive.as_retriever(
    search_kwargs={"k": 3}
)


class SimpleHybridRetriever:
    def __init__(self, bm25_retriever, vector_retriever, k=3):
        self.bm25_retriever = bm25_retriever
        self.vector_retriever = vector_retriever
        self.k = k

    def invoke(self, query):
        bm25_docs = self.bm25_retriever.invoke(query)
        vector_docs = self.vector_retriever.invoke(query)

        unique_docs = {}

        for doc in bm25_docs + vector_docs:
            key = doc.metadata.get("chunk_id", doc.page_content[:100])
            if key not in unique_docs:
                unique_docs[key] = doc

        return list(unique_docs.values())[:self.k]


hybrid_retriever = SimpleHybridRetriever(
    bm25_retriever=bm25_retriever,
    vector_retriever=vector_retriever,
    k=3
)


query = "Какие документы нужны для оформления кредита?"

hybrid_results = hybrid_retriever.invoke(query)

for i, doc in enumerate(hybrid_results, start=1):
    print(f"Hybrid результат {i}")
    print("doc_id:", doc.metadata.get("doc_id"))
    print("title:", doc.metadata.get("title"))
    print("product_type:", doc.metadata.get("product_type"))
    print("text:", doc.page_content)
    print("-" * 100)

Hybrid результат 1
doc_id: borrower_requirements_001
title: Требования к заёмщикам
product_type: borrower_requirements
text: Заёмщиком может быть гражданин Российской Федерации в возрасте от 21 года до 70 лет на дату окончания кредита. Для оформления кредита необходим паспорт, СНИЛС и документ, подтверждающий доход. Минимальный трудовой стаж на текущем месте работы должен составлять не менее 3 месяцев. Общий трудовой стаж должен быть не менее 1 года. Для суммы кредита свыше 1 500 000 рублей банк может запросить копию трудовой книжки или выписку из электронной трудовой книжки
----------------------------------------------------------------------------------------------------
Hybrid результат 2
doc_id: deposit_001
title: Депозитные продукты
product_type: deposit
text: Банк предлагает вклад «Надёжный доход» для физических лиц. Минимальная сумма вклада составляет 10 000 рублей. Срок размещения средств — 3, 6, 12 или 24 месяца. Процентная ставка зависит от срока вклада: 7,5% годовых на 3 ме

Первый результат ну чисто конфетка.

Реализуем фильтрацию по метаданным, чтобы можео было искать среди документов только одного типа, например. В нашем случае это не прям, чтобы очень нужно как будто, но попробуем.

In [24]:
def filtered_similarity_search(vectorstore, query, product_type, k=3, fetch_k=10):
    raw_results = vectorstore.similarity_search(query, k=fetch_k)

    filtered_results = [
        doc for doc in raw_results
        if doc.metadata.get("product_type") == product_type
    ]

    return filtered_results[:k]


query = "Какая процентная ставка и срок?"

filtered_results = filtered_similarity_search(
    vectorstore=vectorstore_recursive,
    query=query,
    product_type="mortgage",
    k=3
)

for i, doc in enumerate(filtered_results, start=1):
    print(f"Filtered result {i}")
    print("doc_id:", doc.metadata.get("doc_id"))
    print("title:", doc.metadata.get("title"))
    print("product_type:", doc.metadata.get("product_type"))
    print("text:", doc.page_content)
    print("-" * 100)

Filtered result 1
doc_id: mortgage_001
title: Ипотечный кредит
product_type: mortgage
text: Ипотечный кредит предоставляется на покупку квартиры, апартаментов или жилого дома. Минимальная сумма ипотеки составляет 500 000 рублей, максимальная сумма — 30 000 000 рублей. Первоначальный взнос — от 15% стоимости недвижимости. Срок кредитования — от 3 до 30 лет. Базовая процентная ставка составляет от 10,5% годовых. Для зарплатных клиентов действует скидка 0,5 процентного пункта. Обязательным условием является страхование объекта недвижимости
----------------------------------------------------------------------------------------------------
Filtered result 2
doc_id: mortgage_001
title: Ипотечный кредит
product_type: mortgage
text: . Обязательным условием является страхование объекта недвижимости. Страхование жизни и здоровья заёмщика не является обязательным, но влияет на итоговую ставку.
----------------------------------------------------------------------------------------------------


Дальше делаем контекстное сжатие, чтобы передавать в модель не весь чанк, а только релевантное предложение. Пока длеаем версию без ЛЛМ, посмотрим получится ли хоть что-то.

In [25]:
def simple_context_compression(docs, query):
    query_terms = set(
        re.sub(r"[^\w\s]", " ", query.lower()).split()
    )

    compressed_docs = []

    for doc in docs:
        sentences = re.split(r"(?<=[.!?])\s+", doc.page_content)
        selected_sentences = []

        for sentence in sentences:
            sentence_terms = set(
                re.sub(r"[^\w\s]", " ", sentence.lower()).split()
            )

            if query_terms & sentence_terms:
                selected_sentences.append(sentence)

        compressed_text = " ".join(selected_sentences).strip()

        if not compressed_text:
            compressed_text = doc.page_content

        compressed_docs.append(
            Document(
                page_content=compressed_text,
                metadata={
                    **doc.metadata,
                    "compressed": True
                }
            )
        )

    return compressed_docs


query = "Какая неустойка при просрочке платежа?"
docs = hybrid_retriever.invoke(query)
compressed_docs = simple_context_compression(docs, query)

for i, doc in enumerate(compressed_docs, start=1):
    print(f"Compressed result {i}")
    print("doc_id:", doc.metadata.get("doc_id"))
    print("title:", doc.metadata.get("title"))
    print("text:", doc.page_content)
    print("-" * 100)

Compressed result 1
doc_id: faq_001
title: FAQ по банковским продуктам
text: Что будет при просрочке платежа по кредиту? При просрочке начисляется неустойка 0,1% от суммы просроченного платежа за каждый день. Можно ли изменить дату платежа? Да, клиент может один раз в год изменить дату ежемесячного платежа
----------------------------------------------------------------------------------------------------
Compressed result 2
doc_id: faq_001
title: FAQ по банковским продуктам
text: Можно ли изменить дату платежа? Да, клиент может один раз в год изменить дату ежемесячного платежа.
----------------------------------------------------------------------------------------------------
Compressed result 3
doc_id: borrower_requirements_001
title: Требования к заёмщикам
text: . Клиент должен иметь постоянную или временную регистрацию на территории региона присутствия банка.
----------------------------------------------------------------------------------------------------


Теперь будем оценивать ретрив. Для этого создаем набор вопросов.

Для каждого вопроса так же указываем документ, в котором находится правильный ответ. Это поможет автоматичеки считать метрики.

In [26]:
test_questions = [
    {
        "question": "Какая минимальная сумма потребительского кредита?",
        "expected_doc_id": "credit_cash_001"
    },
    {
        "question": "Какая максимальная сумма потребительского кредита?",
        "expected_doc_id": "credit_cash_001"
    },
    {
        "question": "На какой срок можно взять потребительский кредит?",
        "expected_doc_id": "credit_cash_001"
    },
    {
        "question": "Можно ли досрочно погасить потребительский кредит без комиссии?",
        "expected_doc_id": "credit_cash_001"
    },

    {
        "question": "Какой первоначальный взнос по ипотеке?",
        "expected_doc_id": "mortgage_001"
    },
    {
        "question": "Какая максимальная сумма ипотечного кредита?",
        "expected_doc_id": "mortgage_001"
    },
    {
        "question": "На какой срок можно оформить ипотеку?",
        "expected_doc_id": "mortgage_001"
    },
    {
        "question": "Обязательно ли страхование недвижимости при ипотеке?",
        "expected_doc_id": "mortgage_001"
    },

    {
        "question": "Какая минимальная сумма вклада?",
        "expected_doc_id": "deposit_001"
    },
    {
        "question": "Какая ставка по вкладу на 12 месяцев?",
        "expected_doc_id": "deposit_001"
    },
    {
        "question": "Можно ли пополнять вклад?",
        "expected_doc_id": "deposit_001"
    },
    {
        "question": "Предусмотрено ли частичное снятие по вкладу?",
        "expected_doc_id": "deposit_001"
    },

    {
        "question": "С какого возраста можно быть заёмщиком?",
        "expected_doc_id": "borrower_requirements_001"
    },
    {
        "question": "Какие документы нужны для оформления кредита?",
        "expected_doc_id": "borrower_requirements_001"
    },
    {
        "question": "Какой минимальный стаж нужен на текущем месте работы?",
        "expected_doc_id": "borrower_requirements_001"
    },
    {
        "question": "Нужна ли регистрация в регионе присутствия банка?",
        "expected_doc_id": "borrower_requirements_001"
    },

    {
        "question": "Можно ли открыть вклад онлайн?",
        "expected_doc_id": "faq_001"
    },
    {
        "question": "Что будет при просрочке платежа по кредиту?",
        "expected_doc_id": "faq_001"
    },
    {
        "question": "Можно ли изменить дату платежа?",
        "expected_doc_id": "faq_001"
    },
    {
        "question": "Где получить справку об остатке задолженности?",
        "expected_doc_id": "faq_001"
    }
]

len(test_questions)

20

Прописываем функции метрик.

Hit Rate@k показывает, попал ли правильный документ в топ-k результатов.
MRR показывает, насколько высоко в выдаче находится первый правильный результат.

In [27]:
def hit_rate_at_k(retriever, test_questions, k=3):
    hits = 0

    for item in test_questions:
        docs = retriever.invoke(item["question"])[:k]
        retrieved_doc_ids = [doc.metadata.get("doc_id") for doc in docs]

        if item["expected_doc_id"] in retrieved_doc_ids:
            hits += 1

    return hits / len(test_questions)


def mean_reciprocal_rank(retriever, test_questions, k=5):
    reciprocal_ranks = []

    for item in test_questions:
        docs = retriever.invoke(item["question"])[:k]
        retrieved_doc_ids = [doc.metadata.get("doc_id") for doc in docs]

        rank = 0

        for i, doc_id in enumerate(retrieved_doc_ids, start=1):
            if doc_id == item["expected_doc_id"]:
                rank = i
                break

        if rank > 0:
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)

    return sum(reciprocal_ranks) / len(reciprocal_ranks)

Создаем несколько ретриверов для сравнения: векторный поиск, векторный поиск с разнообразием, поиск по ключам и гибридный метод.

In [28]:
similarity_retriever = vectorstore_recursive.as_retriever(
    search_kwargs={"k": 5}
)

mmr_retriever_eval = vectorstore_recursive.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 10
    }
)

bm25_retriever_eval = BM25Retriever.from_documents(chunks_recursive)
bm25_retriever_eval.k = 5

hybrid_retriever_eval = SimpleHybridRetriever(
    bm25_retriever=bm25_retriever_eval,
    vector_retriever=similarity_retriever,
    k=5
)

retrievers = {
    "similarity": similarity_retriever,
    "mmr": mmr_retriever_eval,
    "bm25": bm25_retriever_eval,
    "hybrid": hybrid_retriever_eval
}

Ну и считаем метрики.

In [29]:
retrieval_metrics = []

for retriever_name, retriever in retrievers.items():
    retrieval_metrics.append({
        "retriever": retriever_name,
        "HitRate@3": round(hit_rate_at_k(retriever, test_questions, k=3), 3),
        "MRR@5": round(mean_reciprocal_rank(retriever, test_questions, k=5), 3)
    })

retrieval_metrics_df = pd.DataFrame(retrieval_metrics)
retrieval_metrics_df

,retriever,HitRate@3,MRR@5
0,similarity,1.00,0.925
1,mmr,1.00,0.925
2,bm25,0.85,0.693
3,hybrid,0.85,0.693


По результатам оценки лучшими оказались similarity search и MMR: оба подхода получили HitRate@3 = 1.00 и MRR@5 = 0.925. Значит эталонный документ для каждого вопроса попадал в топ-3 результатов, а средняя позиция первого правильного результата была высокой.

BM25 показал более низкое качество: HitRate@3 = 0.85 и MRR@5 = 0.693. Это свзано с его спецификой работы по ключам.


Гибридный поиск показал те метрики, потому что у меня не работал Essembler и я прописывала все руками, поэтому как бы делался снаала BM25 поиск, а потом сверху накладывался векторный. Если BM25 поиск показал плохой результат, то и вектор сверху тоже ничнго хорошего не покажет.


В рамках данного набора данных наиболее эффективным оказался семантический поиск на основе эмбеддингов.

Для дальнейшего улучшения гибрида можно изменить стратегию объединения результатов: например, чередовать результаты BM25 и векторный поиск, использовать взвешенное ранжирование или применять дополнительный reranking.

Посмотрим ошибки.

In [30]:
def analyze_retriever_errors(retriever, test_questions, k=3):
    errors = []

    for item in test_questions:
        docs = retriever.invoke(item["question"])[:k]
        retrieved_doc_ids = [doc.metadata.get("doc_id") for doc in docs]

        if item["expected_doc_id"] not in retrieved_doc_ids:
            errors.append({
                "question": item["question"],
                "expected_doc_id": item["expected_doc_id"],
                "retrieved_doc_ids": retrieved_doc_ids
            })

    return pd.DataFrame(errors)


hybrid_errors = analyze_retriever_errors(
    hybrid_retriever_eval,
    test_questions,
    k=3
)

hybrid_errors

,question,expected_doc_id,retrieved_doc_ids
0,На какой срок можно взять потребительский кредит?,credit_cash_001,"[faq_001, faq_001, borrower_requirements_001]"
1,На какой срок можно оформить ипотеку?,mortgage_001,"[faq_001, faq_001, borrower_requirements_001]"
2,Можно ли пополнять вклад?,deposit_001,"[faq_001, faq_001, borrower_requirements_001]"


Ну тут как бы подтверждение, что гибрид не сработал, пояснение почему --- выше.

##Этап 3

На этом этапе будем интегрировать LLM. Я не буду ничего по апишкам подтягивать (как с ОпенАИ получилось), а буду работать с локальной моделью.

In [31]:
!pip install -q transformers accelerate sentencepiece

После этого подгружаю модель. Лламу не хотелось, поэтому взяла кое-что другое --- google/flan-t5-base. Она маленькая, хорошо работает в колабе, не требует много VRAM и подходит для генерации по инструкции.

In [34]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
llm_model = llm_model.to(device)

print("Используем устройство:", device)
print("LLM успешно загружена.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Используем устройство: cuda
LLM успешно загружена.


Задаем системный промпт для нашей модели.

In [35]:
SYSTEM_PROMPT = """
Ты банковский консультант.

Отвечай только на основе предоставленного контекста.

Если в контексте нет информации для ответа,
скажи:
"В базе знаний нет достаточной информации для ответа."

Не придумывай банковские условия и не добавляй информацию от себя.

Отвечай кратко, понятно и по-русски.

В конце ответа укажи источники.
"""

Реализуем функцию сборки контекста, которая объединяет найденные документы в единый контекст и добавляем информацию об источнике.

In [36]:
def build_context(docs):
    context_parts = []

    for doc in docs:
        context_parts.append(
            f"""
Источник: {doc.metadata.get("title")}
doc_id: {doc.metadata.get("doc_id")}

Текст:
{doc.page_content}
"""
        )

    return "\n\n".join(context_parts)

Функция генерации ответа.

In [39]:
def generate_answer(prompt, max_new_tokens=256, temperature=0.2):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=False
        )

    answer = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return answer

Делаем базовую цепочку для RAG, т.е. ретривал, сбор контекса и генерация ответа

In [37]:
def rag_answer(question, retriever):
    docs = retriever.invoke(question)
    context = build_context(docs)

    prompt = f"""
{SYSTEM_PROMPT}

Контекст:
{context}

Вопрос пользователя:
{question}

Ответ:
"""

    answer = generate_answer(prompt)

    return {
        "question": question,
        "answer": answer,
        "documents": docs
    }

Проверим работает ли.

In [40]:
question = "Какая максимальная сумма ипотечного кредита?"

result = rag_answer(
    question=question,
    retriever=similarity_retriever
)

print("ВОПРОС:")
print(result["question"])

print("\nОТВЕТ:")
print(result["answer"])

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ВОПРОС:
Какая максимальная сумма ипотечного кредита?

ОТВЕТ:
 анковски консултант. твеа толко на основе редоставленноо составленноо составленноо составленноо составленноо составленноо составленноо составленноо составленноо составленноо составленноо составленноо составленноо составленноо составленноо составленноо составленноо составленноо составленноо составленноо составленноо составл


Я попробовала еще несколько локальных гугловских моделей, но это кринж полный, поэтому пробуем что-нибудь другое.


In [65]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

if not torch.cuda.is_available():
    llm_model = llm_model.to("cpu")

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Модель загружена:", model_name)
print("Устройство:", device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Модель загружена: Qwen/Qwen2.5-0.5B-Instruct
Устройство: cuda


In [66]:
def generate_answer_local(system_prompt, user_prompt, max_new_tokens=250):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(llm_model.device)

    with torch.no_grad():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return answer.strip()

In [67]:
SYSTEM_PROMPT = """
Ты — банковский консультант.

Отвечай только на основе предоставленного контекста.
Не придумывай банковские условия, ставки, комиссии и требования.

Если ответа нет в контексте, напиши:
"В базе знаний нет достаточной информации для ответа."

Отвечай кратко, понятно и на русском языке.

В конце ответа укажи источник в формате:
Источник: название документа (doc_id)
"""

In [68]:
def rag_answer(question, retriever, k=3):
    docs = retriever.invoke(question)[:k]

    if not docs:
        return {
            "question": question,
            "answer": "В базе знаний нет достаточной информации для ответа.",
            "documents": []
        }

    context = build_context(docs)

    user_prompt = f"""
Контекст:
{context}

Вопрос пользователя:
{question}

Сформулируй ответ на русском языке только по контексту.
"""

    answer = generate_answer_local(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=user_prompt,
        max_new_tokens=250
    )

    return {
        "question": question,
        "answer": answer,
        "documents": docs
    }

In [69]:
result = rag_answer(
    question="Какая максимальная сумма ипотечного кредита?",
    retriever=similarity_retriever,
    k=3
)

print(result["answer"])

Максимальная сумма ипотечного кредита составляет 30 000 000 рублей.


Фух, наконец-то. Я моделей 6 протестировала, с HF включительно, не хотели работать.

##Этап 4

Анализ и оптимизация. Импортируем то, чего нам не хватает и за работу.

In [70]:
import time
import numpy as np

Создаем тестовые вопросы для RAGa.

In [71]:
rag_eval_questions = [
    {
        "question": "Какая максимальная сумма ипотечного кредита?",
        "expected_doc_id": "mortgage_001"
    },
    {
        "question": "Какая минимальная сумма потребительского кредита?",
        "expected_doc_id": "credit_cash_001"
    },
    {
        "question": "Какая ставка по вкладу на 12 месяцев?",
        "expected_doc_id": "deposit_001"
    },
    {
        "question": "Какие документы нужны для оформления кредита?",
        "expected_doc_id": "borrower_requirements_001"
    },
    {
        "question": "Можно ли досрочно погасить кредит?",
        "expected_doc_id": "faq_001"
    }
]

len(rag_eval_questions)

5

Реализуем функцтю relevant context, которая покажет попал ли RAG в ожидаемый документ.

In [72]:
def evaluate_context_relevancy(question_items, retriever, k=3):
    results = []

    for item in question_items:
        docs = retriever.invoke(item["question"])[:k]
        retrieved_doc_ids = [doc.metadata.get("doc_id") for doc in docs]

        is_relevant = item["expected_doc_id"] in retrieved_doc_ids

        results.append({
            "question": item["question"],
            "expected_doc_id": item["expected_doc_id"],
            "retrieved_doc_ids": retrieved_doc_ids,
            "context_relevant": is_relevant
        })

    return pd.DataFrame(results)

Проверяем правильности источника.

In [73]:
context_relevancy_df = evaluate_context_relevancy(
    question_items=rag_eval_questions,
    retriever=similarity_retriever,
    k=3
)

context_relevancy_df

,question,expected_doc_id,retrieved_doc_ids,context_relevant
0,Какая максимальная сумма ипотечного кредита?,mortgage_001,"[mortgage_001, credit_cash_001, borrower_requi...",True
1,Какая минимальная сумма потребительского кредита?,credit_cash_001,"[credit_cash_001, mortgage_001, borrower_requi...",True
2,Какая ставка по вкладу на 12 месяцев?,deposit_001,"[deposit_001, credit_cash_001, mortgage_001]",True
3,Какие документы нужны для оформления кредита?,borrower_requirements_001,"[borrower_requirements_001, credit_cash_001, b...",True
4,Можно ли досрочно погасить кредит?,faq_001,"[faq_001, credit_cash_001, faq_001]",True


Результаты ну хорошие, так что дальше для корректности считаем долю ответов с правильным источником.

In [74]:
context_relevancy_score = context_relevancy_df["context_relevant"].mean()

print(f"Context relevancy: {context_relevancy_score:.3f}")

Context relevancy: 1.000


Faithfulness проверяем через числовые факты.
Если модель упоминает числа, проценты или суммы, проверяем, встречаются ли эти значения в retrieved context.

In [75]:
def extract_numeric_claims(text):
    return re.findall(r"\d+[,\d\s]*%?|\d+[.,]\d+%?", text)


def evaluate_faithfulness(answer, docs):
    context = " ".join([doc.page_content for doc in docs])

    claims = extract_numeric_claims(answer)

    if not claims:
        return {
            "numeric_claims": [],
            "grounded_claims": [],
            "ungrounded_claims": [],
            "faithfulness": None
        }

    grounded = []
    ungrounded = []

    for claim in claims:
        claim_clean = claim.strip()

        if claim_clean in context:
            grounded.append(claim_clean)
        else:
            ungrounded.append(claim_clean)

    faithfulness = len(grounded) / len(claims) if claims else None

    return {
        "numeric_claims": claims,
        "grounded_claims": grounded,
        "ungrounded_claims": ungrounded,
        "faithfulness": faithfulness
    }

Answer relevancy в MVP-версии считаем через overlap слов вопроса и ответа.

In [76]:
def evaluate_answer_relevancy(question, answer):
    question_terms = set(
        re.sub(r"[^\w\s]", " ", question.lower()).split()
    )

    answer_terms = set(
        re.sub(r"[^\w\s]", " ", answer.lower()).split()
    )

    if not question_terms:
        return 0

    overlap = question_terms & answer_terms

    return len(overlap) / len(question_terms)

Прогоняем RAG по нескольким вопросам и сохрнаяеми ответ, сурс, релевантность контекста, faithfullness, релевантность ответа.

In [77]:
def evaluate_rag_system(question_items, retriever, k=3):
    rows = []

    for item in question_items:
        result = rag_answer(
            question=item["question"],
            retriever=retriever,
            k=k
        )

        answer = result["answer"]
        docs = result["documents"]

        retrieved_doc_ids = [doc.metadata.get("doc_id") for doc in docs]
        context_relevant = item["expected_doc_id"] in retrieved_doc_ids

        faithfulness_report = evaluate_faithfulness(answer, docs)
        answer_relevancy = evaluate_answer_relevancy(item["question"], answer)

        rows.append({
            "question": item["question"],
            "answer": answer,
            "expected_doc_id": item["expected_doc_id"],
            "retrieved_doc_ids": retrieved_doc_ids,
            "context_relevant": context_relevant,
            "faithfulness": faithfulness_report["faithfulness"],
            "ungrounded_claims": faithfulness_report["ungrounded_claims"],
            "answer_relevancy": round(answer_relevancy, 3)
        })

    return pd.DataFrame(rows)

Запускаем.

In [78]:
rag_eval_df = evaluate_rag_system(
    question_items=rag_eval_questions,
    retriever=similarity_retriever,
    k=3
)

rag_eval_df

,question,answer,expected_doc_id,retrieved_doc_ids,context_relevant,faithfulness,ungrounded_claims,answer_relevancy
0,Какая максимальная сумма ипотечного кредита?,Максимальная сумма ипотечного кредита составля...,mortgage_001,"[mortgage_001, credit_cash_001, borrower_requi...",True,1.000000,[],0.800
1,Какая минимальная сумма потребительского кредита?,Минимальная сумма потребительского кредита сос...,credit_cash_001,"[credit_cash_001, mortgage_001, borrower_requi...",True,1.000000,[],0.800
2,Какая ставка по вкладу на 12 месяцев?,"8,2% годовых",deposit_001,"[deposit_001, credit_cash_001, mortgage_001]",True,1.000000,[],0.000
3,Какие документы нужны для оформления кредита?,Для оформления кредита необходимо предоставить...,borrower_requirements_001,"[borrower_requirements_001, credit_cash_001, b...",True,0.666667,[1500000],0.667
4,Можно ли досрочно погасить кредит?,Досрочное погашение кредитов доступно без коми...,faq_001,"[faq_001, credit_cash_001, faq_001]",True,NaN,[],0.000


Интересно, что на 4 вопрос ответ-то правильный, но стоит ноль.

Релевантность ответа оказалась равной 0, поскольку слова «досрочно» и «досрочное», «погасить» и «погашение», «кредит» и «кредитов» не совпадают прям дословно.


Для более точной оценки качества ответов можно использовать лемматизацию, sentence embeddings, BERTScore, Ragas или TruLens.

Считаем агрегированные метрики.

In [79]:
mean_context_relevancy = rag_eval_df["context_relevant"].mean()
mean_answer_relevancy = rag_eval_df["answer_relevancy"].mean()
mean_faithfulness = rag_eval_df["faithfulness"].dropna().mean()

summary_metrics = pd.DataFrame([
    {
        "metric": "context_relevancy",
        "value": round(mean_context_relevancy, 3)
    },
    {
        "metric": "answer_relevancy",
        "value": round(mean_answer_relevancy, 3)
    },
    {
        "metric": "faithfulness_numeric",
        "value": round(mean_faithfulness, 3)
    }
])

summary_metrics

,metric,value
0,context_relevancy,1.000
1,answer_relevancy,0.453
2,faithfulness_numeric,0.917


answer_relevancy просела как раз из-за нуля в четвертом вопросе.

Дальше будем оптимизировать. Начнем с производительности. Измерим время ответа без кеширования.

In [80]:
def measure_response_time(question, retriever, k=3):
    start_time = time.time()

    result = rag_answer(
        question=question,
        retriever=retriever,
        k=k
    )

    end_time = time.time()

    return {
        "question": question,
        "answer": result["answer"],
        "response_time_sec": end_time - start_time
    }


timing_results = []

for item in rag_eval_questions:
    timing_results.append(
        measure_response_time(
            question=item["question"],
            retriever=similarity_retriever,
            k=3
        )
    )

timing_df = pd.DataFrame(timing_results)
timing_df

,question,answer,response_time_sec
0,Какая максимальная сумма ипотечного кредита?,Максимальная сумма ипотечного кредита составля...,0.823376
1,Какая минимальная сумма потребительского кредита?,Минимальная сумма потребительского кредита сос...,0.665239
2,Какая ставка по вкладу на 12 месяцев?,"8,2% годовых",0.219000
3,Какие документы нужны для оформления кредита?,Для оформления кредита необходимы следующие до...,3.727843
4,Можно ли досрочно погасить кредит?,Досрочное погашение кредитов доступно без коми...,0.582308


По моему ну нормально, один вопрос только в 3 секундах, но имхо норм. Среднее время посмотрим тоже, но из-за 3.7 секунд скорее всего подпортится.

In [81]:
mean_time_without_cache = timing_df["response_time_sec"].mean()

print(f"Среднее время ответа без кеша: {mean_time_without_cache:.3f} сек.")

Среднее время ответа без кеша: 1.204 сек.


Будем считать, что нормально.

Реализуем простой кеш для ретривал.

In [82]:
retrieval_cache = {}

def cached_retrieve(question, retriever, k=3):
    cache_key = (question, k)

    if cache_key in retrieval_cache:
        return retrieval_cache[cache_key]

    docs = retriever.invoke(question)[:k]
    retrieval_cache[cache_key] = docs

    return docs

RAG с кешированием, т.е. при повторных запросах ретривал ускоряется. Ответ ЛЛЛМ выполняется каждый раз.

In [83]:
def rag_answer_cached(question, retriever, k=3):
    docs = cached_retrieve(question, retriever, k=k)

    if not docs:
        return {
            "question": question,
            "answer": "В базе знаний нет достаточной информации для ответа.",
            "documents": []
        }

    context = build_context(docs)

    user_prompt = f"""
Контекст:
{context}

Вопрос пользователя:
{question}

Сформулируй ответ на русском языке только по контексту.
"""

    answer = generate_answer_local(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=user_prompt,
        max_new_tokens=250
    )

    return {
        "question": question,
        "answer": answer,
        "documents": docs
    }

Измеряем время с кешем.

In [84]:
#cначала прогреваем кеш: один раз вызываем retrieval для всех вопросов
for item in rag_eval_questions:
    _ = cached_retrieve(
        question=item["question"],
        retriever=similarity_retriever,
        k=3
    )


#измеряем время ответа с уже заполненным ретривал-кешем
def measure_response_time_cached(question, retriever, k=3):
    start_time = time.time()

    result = rag_answer_cached(
        question=question,
        retriever=retriever,
        k=k
    )

    end_time = time.time()

    return {
        "question": question,
        "answer": result["answer"],
        "response_time_sec": end_time - start_time
    }


cached_timing_results = []

for item in rag_eval_questions:
    cached_timing_results.append(
        measure_response_time_cached(
            question=item["question"],
            retriever=similarity_retriever,
            k=3
        )
    )

cached_timing_df = pd.DataFrame(cached_timing_results)
cached_timing_df

,question,answer,response_time_sec
0,Какая максимальная сумма ипотечного кредита?,Максимальная сумма ипотечного кредита составля...,0.807648
1,Какая минимальная сумма потребительского кредита?,Минимальная сумма потребительского кредита сос...,0.652210
2,Какая ставка по вкладу на 12 месяцев?,"8,2% годовых",0.217462
3,Какие документы нужны для оформления кредита?,Для оформления кредита необходимо предоставить...,3.572805
4,Можно ли досрочно погасить кредит?,Досрочное погашение кредитов доступно без коми...,0.569832


Сравниваем до кеша и после, а то так неудобно мотать туда сюда.

In [85]:
mean_time_with_cache = cached_timing_df["response_time_sec"].mean()

performance_comparison = pd.DataFrame([
    {
        "mode": "without_cache",
        "mean_response_time_sec": round(mean_time_without_cache, 3)
    },
    {
        "mode": "with_retrieval_cache",
        "mean_response_time_sec": round(mean_time_with_cache, 3)
    }
])

performance_comparison

,mode,mean_response_time_sec
0,without_cache,1.204
1,with_retrieval_cache,1.164


Даже какое-то улучшение незанчительное есть.

Ещё один способ ускорения — уменьшить количество retirived чанков.

In [86]:
timing_k2_results = []

for item in rag_eval_questions:
    timing_k2_results.append(
        measure_response_time(
            question=item["question"],
            retriever=similarity_retriever,
            k=2
        )
    )

timing_k2_df = pd.DataFrame(timing_k2_results)

mean_time_k2 = timing_k2_df["response_time_sec"].mean()

print(f"Среднее время ответа при k=2: {mean_time_k2:.3f} сек.")

Среднее время ответа при k=2: 0.845 сек.


Сравниваем производительность итоговую.

In [87]:
performance_summary = pd.DataFrame([
    {
        "configuration": "k=3, no cache",
        "mean_response_time_sec": round(mean_time_without_cache, 3)
    },
    {
        "configuration": "k=3, retrieval cache",
        "mean_response_time_sec": round(mean_time_with_cache, 3)
    },
    {
        "configuration": "k=2, no cache",
        "mean_response_time_sec": round(mean_time_k2, 3)
    }
])

performance_summary

,configuration,mean_response_time_sec
0,"k=3, no cache",1.204
1,"k=3, retrieval cache",1.164
2,"k=2, no cache",0.845


По-моему получилось легендарно, больше ничего трогать уже не хочу в этом плане.

Ну и теперь можно сделать итоговый вывод по проекту:
1. Победила RAG --- уже хорошо.

На первом этапе я создала набор синтетических документов, которые были очищены и приведины к единому формату, потом разбиты на чанки с помощью трех стратегий и также были созданы эбеддинги.

На втором этапе я реализовала similarity search, MMR, BM25 и гибридный поиск.
По результатам оценки на 20 тестовых вопросах лучше всего показали себя similarity search
и MMR: они достигли HitRate@3 = 1.00 и MRR@5 = 0.925. BM25 и простая гибридная реализация
показали более низкие результаты, что связано с зависимостью BM25 от буквального совпадения слов.

На третьем этапе уже была реализована RAG-цепочка с локальной instruction-моделью
Qwen2.5-0.5B-Instruct. Система извлекает релевантные чанки, формирует контекст,
передаёт его в LLM и возвращает ответ с указанием источников. Также были добавлены
обработка отсутствия релевантной информации, self-query, multi-query, reranking
и проверка соответствия ответа источнику.

На четвёртом этапе была проведена оценка качества RAG-системы.
Context relevancy составила 1.000, что означает попадание правильного источника
в retrieved context для всех тестовых вопросов. Faithfulness_numeric составила 0.917,
то есть большинство числовых утверждений модели подтверждались найденными источниками.
Answer relevancy оказалась ниже — 0.453 — из-за ограничений простой overlap-метрики,
которая не учитывает морфологию русского языка и семантическую близость выражений.

Также была проведена оптимизация производительности: измерено время ответа системы,
реализовано кеширование retrieval-результатов и протестировано уменьшение количества
retrieved chunks. Кеширование позволяет ускорить повторные запросы, а уменьшение k снижает
размер prompt и может ускорять генерацию.